In [1]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph 

load_dotenv()

True

In [6]:
os.getenv('NEO4J_USERNAME')

'a5b4ce5d'

In [4]:
graph = Neo4jGraph()

In [5]:
graph

In [7]:
from langchain_groq import ChatGroq

In [9]:
model = ChatGroq(model="openai/gpt-oss-120b")

In [10]:
response = model.invoke("Hello").content

In [11]:
response

'Hello! How can I assist you today?'

In [12]:
from langchain_core.documents import Document

In [13]:
text="""
    Elon Reeve Musk (born June 28, 1971) is a businessman and investor known for his key roles in space
    company SpaceX and automotive company Tesla, Inc. Other involvements include ownership of X Corp.,
    formerly Twitter, and his role in the founding of The Boring Company, xAI, Neuralink and OpenAI.
    He is one of the wealthiest people in the world; as of July 2024, Forbes estimates his net worth to be
    US$221 billion.Musk was born in Pretoria to Maye and engineer Errol Musk, and briefly attended
    the University of Pretoria before immigrating to Canada at age 18, acquiring citizenship through
    his Canadian-born mother. Two years later, he matriculated at Queen's University at Kingston in Canada.
    Musk later transferred to the University of Pennsylvania and received bachelor's degrees in economics
    and physics. He moved to California in 1995 to attend Stanford University, but dropped out after
    two days and, with his brother Kimbal, co-founded online city guide software company Zip2.
"""

In [17]:
documents = [Document(page_content=text)]
documents[0].page_content

"\n    Elon Reeve Musk (born June 28, 1971) is a businessman and investor known for his key roles in space\n    company SpaceX and automotive company Tesla, Inc. Other involvements include ownership of X Corp.,\n    formerly Twitter, and his role in the founding of The Boring Company, xAI, Neuralink and OpenAI.\n    He is one of the wealthiest people in the world; as of July 2024, Forbes estimates his net worth to be\n    US$221 billion.Musk was born in Pretoria to Maye and engineer Errol Musk, and briefly attended\n    the University of Pretoria before immigrating to Canada at age 18, acquiring citizenship through\n    his Canadian-born mother. Two years later, he matriculated at Queen's University at Kingston in Canada.\n    Musk later transferred to the University of Pennsylvania and received bachelor's degrees in economics\n    and physics. He moved to California in 1995 to attend Stanford University, but dropped out after\n    two days and, with his brother Kimbal, co-founded onli

In [18]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

In [19]:
llm_graph_transformer = LLMGraphTransformer(llm=model)

In [20]:
graph_documents = llm_graph_transformer.convert_to_graph_documents(documents)

In [22]:
graph_documents[0].nodes

[Node(id='Elon Reeve Musk', type='Person', properties={}),
 Node(id='Maye', type='Person', properties={}),
 Node(id='Errol Musk', type='Person', properties={}),
 Node(id='Kimbal', type='Person', properties={}),
 Node(id='Spacex', type='Company', properties={}),
 Node(id='Tesla, Inc.', type='Company', properties={}),
 Node(id='X Corp.', type='Company', properties={}),
 Node(id='Twitter', type='Company', properties={}),
 Node(id='The Boring Company', type='Company', properties={}),
 Node(id='Xai', type='Company', properties={}),
 Node(id='Neuralink', type='Company', properties={}),
 Node(id='Openai', type='Company', properties={}),
 Node(id='Zip2', type='Company', properties={}),
 Node(id='Forbes', type='Organization', properties={}),
 Node(id='University Of Pretoria', type='Organization', properties={}),
 Node(id="Queen'S University At Kingston", type='Organization', properties={}),
 Node(id='University Of Pennsylvania', type='Organization', properties={}),
 Node(id='Stanford University

In [23]:
graph_documents[0].relationships

[Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Maye', type='Person', properties={}), type='PARENT', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Errol Musk', type='Person', properties={}), type='PARENT', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Kimbal', type='Person', properties={}), type='SIBLING', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Spacex', type='Company', properties={}), type='FOUNDED', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='Tesla, Inc.', type='Company', properties={}), type='INVESTOR', properties={}),
 Relationship(source=Node(id='Elon Reeve Musk', type='Person', properties={}), target=Node(id='X Corp.', type='Company', properties={}), type='OWNER', proper

In [24]:
### Load the dataset of movie

movie_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') |
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') |
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

In [25]:
graph.query(movie_query)

[]

In [27]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Person {born: STRING, name: STRING, title: STRING}
Champion {title: STRING, founded: STRING}
Movie {title: STRING, id: STRING, released: DATE, imdbRating: FLOAT}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Person)-[:FIGHTER]->(:Champion)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


In [32]:
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph

In [34]:
chain = GraphCypherQAChain.from_llm(
    llm=model, graph=graph, verbose=True, allow_dangerous_requests=True
)

In [35]:
response=chain.invoke({"query":"Who was the director of the moview GoldenEye"})

response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: 'GoldenEye'})
RETURN p.name AS director;


[#D497]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-3b814dd7cab7-15-0050.production-orch-0068.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.171.25', 7687))): OSError('No data')
Transaction failed and will be retried in 1.026707076873087s (Failed to read from defunct connection IPv4Address(('p-mt-3b814dd7cab7-15-0050.production-orch-0068.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.171.25', 7687))))


Full Context:
[{'director': 'Martin Campbell'}]

> Finished chain.


{'query': 'Who was the director of the moview GoldenEye',
 'result': 'Martin Campbell.'}

In [36]:
response=chain.invoke({"query":"Who was the director in movie Casino"})

response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: 'Casino'})
RETURN p.name AS director
Full Context:
[{'director': 'Martin Scorsese'}]

> Finished chain.


{'query': 'Who was the director in movie Casino',
 'result': 'Martin Scorsese was the director of the movie Casino.'}

In [37]:
response=chain.invoke({"query":"Which movie were released in 2008"})

response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie)
WHERE m.released.year = 2008
RETURN m.title
Full Context:
[]

> Finished chain.


{'query': 'Which movie were released in 2008',
 'result': 'I don’t know the answer.'}

In [38]:
response=chain.invoke({"query":"Give me the list of movie having imdb rating more than 8"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie) 
WHERE m.imdbRating > 8 
RETURN m.title, m.imdbRating;
Full Context:
[{'m.title': 'Toy Story', 'm.imdbRating': 8.3}, {'m.title': 'Heat', 'm.imdbRating': 8.2}, {'m.title': 'Casino', 'm.imdbRating': 8.2}, {'m.title': 'Twelve Monkeys (a.k.a. 12 Monkeys)', 'm.imdbRating': 8.1}, {'m.title': 'Seven (a.k.a. Se7en)', 'm.imdbRating': 8.6}, {'m.title': 'Usual Suspects, The', 'm.imdbRating': 8.6}, {'m.title': 'Hate (Haine, La)', 'm.imdbRating': 8.1}, {'m.title': 'Braveheart', 'm.imdbRating': 8.4}, {'m.title': 'Taxi Driver', 'm.imdbRating': 8.3}, {'m.title': 'Anne Frank Remembered', 'm.imdbRating': 8.2}]

> Finished chain.


{'query': 'Give me the list of movie having imdb rating more than 8',
 'result': 'Toy Story, Heat, Casino, Twelve Monkeys (a.k.a. 12 Monkeys), Seven (a.k.a. Se7en), Usual Suspects, The, Hate (Haine, La), Braveheart, Taxi Driver, Anne Frank Remembered.'}